# 02 — Aedes Puerto Rico schema (synthetic proxy)

**Labels:** `[OPERACIONAL]` synthetic stand-in.  
**Not** an empirical claim about Caño Martín Peña vs Candelaria until licensed field data land in `data/aedes/`.

When real trap CSVs are available: swap only the loader cell.


## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "python" / "core").is_dir():
    pass
elif (ROOT.parent / "python" / "core").is_dir():
    ROOT = ROOT.parent
else:
    # walk up
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "python" / "core").is_dir():
            ROOT = p
            break

sys.path.insert(0, str(ROOT / "python"))
from core import (
    THETA_CHAOS, THETA_STABLE, DELTA,
    compute_taus, accumulate_time, gate_function,
)
print("ROOT =", ROOT)
print("δ =", DELTA, "τ_ch =", THETA_CHAOS, "τ_st =", THETA_STABLE)

## Synthetic two-site panel


In [ ]:
def synthetic_two_sites(T=200, seed=1):
    """Two sites × 3 traps; site B gains volatility after t=120."""
    rng = np.random.default_rng(seed)
    t = np.arange(T)
    seasonal = 1.2 + np.sin(2 * np.pi * t / 52)
    site_a = np.column_stack([seasonal + 0.3 * rng.normal(size=T) for _ in range(3)])
    vol = np.where(t > 120, 1.5, 0.3)
    site_b = np.column_stack([seasonal * 0.8 + vol * rng.normal(size=T) for _ in range(3)])
    return {
        "Cano_Martin_Pena_proxy": np.maximum(site_a, 0.0),
        "Candelaria_proxy": np.maximum(site_b, 0.0),
    }

def summarize(name, X):
    tg, _ = compute_taus(X, window_size=13)
    T, _, g, depth = accumulate_time(tg, window_size=13)
    valid = tg[~np.isnan(tg)]
    return {
        "name": name, "X": X, "tau": tg, "T": T, "g": g,
        "mean_tau": float(valid.mean()),
        "T_end": float(T[-1]),
        "max_k": int(depth.max()),
    }

sites = synthetic_two_sites()
summaries = [summarize(n, X) for n, X in sites.items()]
for s in summaries:
    print(f"{s['name']}: mean τₛ={s['mean_tau']:.3f}  T_RECD={s['T_end']:.5f}  max_k={s['max_k']}")

## Figures


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for col, s in enumerate(summaries):
    t = np.arange(len(s["tau"]))
    axes[0, col].plot(t, s["tau"], lw=0.9)
    axes[0, col].axhline(THETA_CHAOS, color="C3", ls="--", lw=0.8)
    axes[0, col].axhline(-THETA_CHAOS, color="C3", ls="--", lw=0.8)
    axes[0, col].set_title(f"τₛ — {s['name']}")
    axes[0, col].set_ylabel("τₛ")
    axes[1, col].plot(t, s["T"], lw=1.0, color="C4")
    axes[1, col].set_title("T_RECD")
    axes[1, col].set_xlabel("time index")
    axes[1, col].set_ylabel("T")
fig.suptitle("[OPERACIONAL] Synthetic Aedes-schema proxy (not field validation)", y=1.02)
fig.tight_layout()
plt.show()

## Expected CSV schema (when real data arrive)

- Rows: time (weeks or trap visits)  
- Columns: trap / module counts (N ≥ 2)  
- Place files under `data/aedes/` with license note in `data/aedes/README.md`  
- Then load with `np.genfromtxt` / pandas and reuse `summarize`.
